# adaptive_fpr criterion comparison: **D / A / L** at a fixed budget

Runs **only `adaptive_fpr`**, once per optimality criterion (D/A/L), each capped at the same
**fixed** total revealed-shot `BUDGET` you set below. No fixed_fpr / fixed_no_fpr. Each criterion's
run is stored in its own folder: `adaptive_D/`, `adaptive_A/`, `adaptive_L/`.

Self-contained (imports `gst_seed_experiment`). **Needs PyROL -> run in Docker.** L is slow (its
infidelity metric does gauge-opts); D and A are fast.

In [ ]:
import sys
from pathlib import Path
candidates = [
    Path.cwd(),
    Path.cwd() / "seed_sweep_experiments",
    Path.cwd() / "GST_POUNDERS" / "seed_sweep_experiments",
    Path("/workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments"),
]
EXPERIMENT_DIR = next(
    (p.resolve() for p in candidates if (p / "gst_seed_experiment.py").exists()), None
)
if EXPERIMENT_DIR is None:
    raise FileNotFoundError("Could not locate gst_seed_experiment.py")
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

from gst_seed_experiment import ExperimentConfig, run_one_experiment
config = ExperimentConfig.from_json(EXPERIMENT_DIR / "experiment_config.json")
print("Experiment directory:", EXPERIMENT_DIR)
print("noise_model:", config.noise_model, "| model_kind:", config.model_kind)

In [ ]:
# ------------------------- knobs -------------------------
SEEDS       = "101:106"                        # e.g. "101:106" for seeds 101-105
CRITERIA    = "D,A,L"                           # any subset of D,A,L
BUDGET      = 1000000                           # FIXED total revealed-shot budget for adaptive
RESULTS_DIR = EXPERIMENT_DIR / "criterion_comparison"
FORCE       = False                             # True -> rerun even completed bundles

In [ ]:
import json
from dataclasses import replace, asdict
import numpy as np
import pandas as pd

INF = "mean_gate_entanglement_infidelity_to_truth"

def parse_seeds(spec):
    seeds = []
    for tok in str(spec).split(","):
        tok = tok.strip()
        if not tok:
            continue
        if ":" not in tok:
            seeds.append(int(tok)); continue
        parts = [int(v) for v in tok.split(":")]
        start, stop = parts[0], parts[1]
        step = parts[2] if len(parts) == 3 else 1
        seeds.extend(range(start, stop, step))
    return sorted(set(seeds))

def _completed(rd):
    return (rd / "completed.json").exists() and (rd / "summary.json").exists()

def _config_matches(rd, cfg):
    p = rd / "config.json"
    return p.exists() and json.loads(p.read_text()) == json.loads(json.dumps(asdict(cfg)))

def run_or_load(cfg, seed, method, rd, force):
    if _completed(rd) and not force and _config_matches(rd, cfg):
        print(f"SKIP seed={seed} {rd.name}: completed")
        return json.loads((rd / "summary.json").read_text())
    print(f"RUN  seed={seed} {rd.name}")
    return run_one_experiment(config=cfg, data_seed=seed, method=method, output_dir=rd)

In [ ]:
# ---- run adaptive_fpr with each criterion at the FIXED budget ----
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
criteria = [c.strip().upper() for c in CRITERIA.split(",") if c.strip()]
rows = []
for seed in parse_seeds(SEEDS):
    seed_dir = RESULTS_DIR / f"seed_{seed:06d}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    for crit in criteria:
        cfg = replace(config, adaptive_criterion=crit, adaptive_total_shot_budget=int(BUDGET))
        try:
            res = run_or_load(cfg, seed, "adaptive_fpr", seed_dir / f"adaptive_{crit}", FORCE)
            rows.append({"seed": seed, "criterion": crit, INF: res[INF],
                         "accounted_revealed_shots": res["accounted_revealed_shots"],
                         "max_shots_per_circuit": res["max_shots_per_circuit"], "flag": res["flag"]})
            print(f"  seed={seed} {crit}: infid={res[INF]:.3e}  "
                  f"shots={res['accounted_revealed_shots']:,}  max/circ={res['max_shots_per_circuit']}")
        except Exception as exc:
            print(f"  seed={seed} {crit}: FAILED {exc!r}")
            rows.append({"seed": seed, "criterion": crit, INF: float("nan"),
                         "accounted_revealed_shots": 0, "max_shots_per_circuit": 0, "flag": "FAILED"})
    pd.DataFrame(rows).to_csv(RESULTS_DIR / "criterion_comparison_summary.csv", index=False)

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "criterion_comparison_summary.csv", index=False)
print("\nsaved", RESULTS_DIR / "criterion_comparison_summary.csv")

In [ ]:
# ---- table: median by criterion ----
print("=== per (seed, criterion) ===")
cols = [c for c in ["seed", "criterion", INF, "accounted_revealed_shots", "max_shots_per_circuit", "flag"] if c in df.columns]
print(df[cols].to_string(index=False))

print("\n=== median by criterion (the headline) ===")
summary = df.groupby("criterion").agg(
    median_infidelity=(INF, "median"),
    median_accounted_shots=("accounted_revealed_shots", "median"),
    median_max_shots_per_circuit=("max_shots_per_circuit", "median"),
    n_seeds=("seed", "count"),
)
print(summary.to_string())

In [ ]:
# ---- plot: D / A / L infidelity per seed ----
import matplotlib.pyplot as plt
crits = [c for c in ["D", "A", "L"] if c in set(df["criterion"])]
seeds = sorted(df["seed"].unique())
colors = {"D": "#D55E00", "A": "#009E73", "L": "#0072B2"}
x = np.arange(len(seeds)); w = 0.8 / max(len(crits), 1)
fig, ax = plt.subplots(figsize=(max(11, 2.2 * len(seeds) + 3), 7))
for i, crit in enumerate(crits):
    vals = [pd.to_numeric(df[(df.seed == s) & (df.criterion == crit)][INF], errors="coerce").mean() for s in seeds]
    ax.bar(x + (i - (len(crits) - 1) / 2) * w, vals, w, label=f"adaptive-{crit}", color=colors.get(crit, "#444"))
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(seeds, fontsize=13)
ax.tick_params(axis="y", labelsize=13)
ax.axhline(1e-4, ls="--", color="black", lw=1.2, label="target 1e-4")
ax.set_xlabel("data seed", fontsize=14); ax.set_ylabel("mean gate infidelity to truth (log)", fontsize=14)
ax.set_title(f"adaptive_fpr: D / A / L at fixed budget = {int(BUDGET):,} shots", fontsize=15)
ax.legend(fontsize=12); fig.tight_layout()
fig.savefig(RESULTS_DIR / "criterion_comparison.png", dpi=150)
plt.show()
print("saved", RESULTS_DIR / "criterion_comparison.png")

## How to read this

- **median_infidelity** across D / A / L at the same fixed budget = the headline (which criterion
  is most accurate for the same number of shots).
- **median_max_shots_per_circuit**: huge for **D** = over-concentration (the failure mode we
  diagnosed). If **A is close to L** on both infidelity and max-shots -> use **A** (no expensive
  metric -> scalable to 2Q).
- **median_accounted_shots** should be ~`BUDGET` for each criterion (confirms they actually spent
  the budget - if it's stuck near the baseline, that criterion did not sample).
- **flag** != -2 -> that run did not converge cleanly; treat with caution.

## Convergence per criterion (infidelity vs POUNDERS iteration)

Overlays D / A / L infidelity-to-truth over the optimization, one panel per seed. Shows *how* each criterion converges (and whether any diverges / plateaus), not just the final number.

In [ ]:
# ==== Convergence: infidelity vs POUNDERS iteration, per criterion ====
import re, math
import matplotlib.pyplot as plt

INF = "mean_gate_entanglement_infidelity_to_truth"
crit_colors = {"D": "#D55E00", "A": "#009E73", "L": "#0072B2"}

seed_dirs = [d for d in sorted(RESULTS_DIR.glob("seed_*")) if list(d.glob("adaptive_*/iteration_accuracy.csv"))]
if not seed_dirs:
    print("No adaptive_*/iteration_accuracy.csv under", RESULTS_DIR, "- run the sweep first.")
else:
    nseed = len(seed_dirs)
    ncols = min(3, nseed)                      # reflow into a grid so each panel is big
    nrows = math.ceil(nseed / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(7.5 * ncols, 6.0 * nrows), squeeze=False)
    axflat = axes.flatten()
    for j, sd in enumerate(seed_dirs):
        ax = axflat[j]
        seed = re.search(r"seed_(\d+)", sd.name).group(1)
        plotted = False
        for crit in ["D", "A", "L"]:
            f = sd / f"adaptive_{crit}" / "iteration_accuracy.csv"
            if not f.exists():
                continue
            d = pd.read_csv(f)
            if INF not in d.columns or len(d) == 0:
                continue
            xcol = "iteration" if "iteration" in d.columns else ("nf" if "nf" in d.columns else None)
            xvals = d[xcol] if xcol else range(len(d))
            ax.plot(xvals, d[INF], color=crit_colors.get(crit, "#444"), label=f"adaptive-{crit}", lw=2.4)
            plotted = True
        ax.set_yscale("log")
        ax.axhline(1e-4, ls="--", color="black", lw=1.2, label="target 1e-4")
        ax.set_xlabel("POUNDERS iteration", fontsize=13)
        ax.set_ylabel("mean gate infidelity to truth (log)", fontsize=13)
        ax.set_title(f"seed {seed}: convergence (D / A / L)", fontsize=14)
        ax.tick_params(labelsize=12)
        if plotted:
            ax.legend(fontsize=11)
        ax.grid(alpha=0.25, which="both")
    for k in range(nseed, len(axflat)):        # hide any empty grid cells
        axflat[k].axis("off")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "criterion_convergence.png", dpi=150)
    plt.show()
    print("saved", RESULTS_DIR / "criterion_convergence.png")